# LTX-Video 2.3 Text-to-Video Generation with ComfyUI on AMD W7900

With the rapid development of AI-generated content (AIGC) technologies, text-to-video generation has become a powerful tool for creators, researchers, and media professionals.

[ComfyUI](https://github.com/comfyanonymous/ComfyUI) is a node-based graphical interface for diffusion models, enabling users to visually construct AI image/video generation workflows through modular operations.

[LTX-Video 2.3](https://github.com/Lightricks/LTX-Video) is Lightricks' latest open-source text-to-video model — a 22B-parameter transformer capable of generating 1280x704 video at 25 fps with high temporal consistency.

This hands-on workshop walks you through running **LTX-Video 2.3** via ComfyUI on **AMD W7900 GPUs using ROCm software**. You will learn how to launch ComfyUI inside this notebook environment and generate videos from text prompts on AMD hardware.


## Prerequisites

This workshop was developed and tested with the following setup:

### Hardware
- **AMD W7900 (gfx1100, 48GB VRAM)** — single GPU is sufficient for LTX-2.3 BF16 inference.

### Software
- **ROCm 7.2.4** + **PyTorch 2.10 (HIP)** — pre-installed in the container image.
- **ComfyUI 0.25.0** — pre-installed at `/comfyui_workspace/ComfyUI`.
- **LTX-2.3 models (BF16)** — mounted at `/comfyui_workspace/ComfyUI/models` at container start.

Confirm ROCm is available:


In [ ]:
!rocm-smi


This command lists available AMD GPUs along with their key details (VRAM, power, temperature).


# ComfyUI Setup

To set up the ComfyUI inference environment, follow the steps below.


#### Verify the PyTorch installation

Verify that PyTorch is correctly installed.


**Step 1** Verify PyTorch is installed and can detect the GPU compute device.


In [ ]:
!python3 -c 'import torch' 2>/dev/null && echo 'Success' || echo 'Failure'


The expected result is `Success`.


**Step 2** Confirm the GPU is available.


In [ ]:
!python3 -c 'import torch; print(torch.cuda.is_available())'


The expected result is `True`.


**Step 3** Display the installed GPU device name.


In [ ]:
!python3 -c "import torch; print(f'device name [0]:', torch.cuda.get_device_name(0))"


The expected result should be similar to: `device name [0]: AMD Radeon PRO W7900`


# ComfyUI Installation

ComfyUI is pre-installed at `/comfyui_workspace/ComfyUI`. Confirm it is ready to use.


In [ ]:
%cd /comfyui_workspace/ComfyUI
!ls main.py && echo 'ComfyUI found OK' || echo 'ComfyUI not found'


In [ ]:
!cat /comfyui_workspace/ComfyUI/comfyui_version.py


# Running Video Generation

Follow these steps to generate video from your text prompts.

---

## Model Setup

### Required Models

LTX-Video 2.3 requires **5 model files** (BF16 full-precision versions for best compatibility with W7900).
These models are pre-mounted into the container — no download is needed.

| Role | File | Directory | Size |
|------|------|-----------|------|
| Main diffusion model | `ltx-2.3-22b-dev.safetensors` | `models/checkpoints/` | 46 GB |
| Text encoder | `gemma_3_12B_it.safetensors` | `models/text_encoders/` | 24 GB |
| Distilled LoRA | `ltx_2.3_22b_distilled_1.1_lora_dynamic_fro09_avg_rank_111_bf16.safetensors` | `models/loras/` | 2.7 GB |
| Gemma LoRA | `gemma-3-12b-it-abliterated_lora_rank64_bf16.safetensors` | `models/loras/` | 0.6 GB |
| Spatial upscaler | `ltx-2.3-spatial-upscaler-x2-1.1.safetensors` | `models/latent_upscale_models/` | 1 GB |

### Directory Structure

```text
ComfyUI/
└── models/
    ├── checkpoints/            ltx-2.3-22b-dev.safetensors
    ├── text_encoders/          gemma_3_12B_it.safetensors
    ├── loras/                  ltx_2.3_22b_distilled_1.1_lora_dynamic_fro09_avg_rank_111_bf16.safetensors
    │                           gemma-3-12b-it-abliterated_lora_rank64_bf16.safetensors
    └── latent_upscale_models/  ltx-2.3-spatial-upscaler-x2-1.1.safetensors
```


### Verify Models Are Present

Run the cell below to confirm all required models exist in the correct directories:


In [ ]:
import os

COMFY_ROOT = "/comfyui_workspace/ComfyUI"

required_models = [
    ("checkpoints",           "ltx-2.3-22b-dev.safetensors"),
    ("text_encoders",         "gemma_3_12B_it.safetensors"),
    ("loras",                 "ltx_2.3_22b_distilled_1.1_lora_dynamic_fro09_avg_rank_111_bf16.safetensors"),
    ("loras",                 "gemma-3-12b-it-abliterated_lora_rank64_bf16.safetensors"),
    ("latent_upscale_models", "ltx-2.3-spatial-upscaler-x2-1.1.safetensors"),
]

all_ok = True
for subdir, fname in required_models:
    path = os.path.join(COMFY_ROOT, "models", subdir, fname)
    exists = os.path.isfile(path)
    size_gb = os.path.getsize(path) / 1e9 if exists else 0
    status = f"OK  {size_gb:.1f} GB" if exists else "MISSING"
    print(f"  [{status}]  {subdir}/{fname}")
    if not exists:
        all_ok = False

print()
print("All models present. Ready to launch ComfyUI." if all_ok else
      "WARNING: Some models are missing. Contact your instructor.")


## Launch the Server

To start the ComfyUI server, run the following command:


In [ ]:
%cd /comfyui_workspace/ComfyUI
!python3 main.py --listen 0.0.0.0 --port 8188


Once you see a message like `To see the GUI go to: http://0.0.0.0:8188` in the terminal output, it means the **ComfyUI server has launched successfully**.

> **Note**: This cell runs indefinitely while the server is active. Do not interrupt it.
>
> Use the **Your App URL** shown on the **Launch Notebook** page to access the interface in your browser.
> Do **not** use `http://0.0.0.0:8188` directly — it is only reachable inside the container.


## Open the ComfyUI Interface

Once you paste the corrected link into your browser, the ComfyUI interface will open.

You will see:
- A **node-based canvas** in the main area
- A **sidebar on the left**, where you can load workflows and start generating

---

## Load the Workflow

ComfyUI workflows define the full pipeline and all parameters required to generate a video. They are saved as JSON files and can be loaded from the sidebar.

In this workshop, we will use the **LTX-2.3_t2v_BF16** workflow.

---

### How to Load the Workflow

Once you have launched the ComfyUI interface in your browser:

1. Click **Workflows** in the sidebar.
2. Select **LTX-2.3_t2v_BF16**.

The full workflow, with all pre-configured nodes, will load onto your canvas and be ready to run.

This setup ensures you can immediately start experimenting with **text-to-video generation** using the **LTX-Video 2.3 22B model** on AMD hardware.


## Run the Workflow

Before running the video generation, make sure the correct models and settings are loaded.

---

### Key Parameters

Review and adjust these parameters in the workflow canvas before running:

| Parameter | Default | Description |
|-----------|---------|-------------|
| **Positive Prompt** | *(your text)* | Describe the video scene to generate |
| **Negative Prompt** | *(optional)* | Elements to avoid in the generation |
| **Resolution** | 1280 x 704 | Output video resolution |
| **Frames** | 121 | Number of frames (~5 sec at 25 fps) |
| **Steps** | 30 | Denoising steps (more = higher quality, slower) |

---

### Execute Video Generation

Click the **Run** button, or press **Ctrl (Cmd) + Enter** to start the video generation process.


## Expected Result

On a single **AMD W7900 (48GB)**, LTX-2.3 BF16 generation takes approximately **3.5 to 4 minutes** and outputs:
- Resolution: **1280 x 704**
- Frame rate: **25 fps**
- Duration: **~5 seconds** (121 frames)

The output video is saved to:
```
/comfyui_workspace/ComfyUI/output/video/
```

You can also download it directly from the ComfyUI interface after generation completes.


## Advanced Assignment: Create an AMD-Themed Video in ComfyUI

### Objective
Your task is to create a short, cinematic video inspired by **AMD's technological vision**, using a customized workflow and prompts of your choice.

---

### Instructions

1. **Choose Your Prompt**

   Edit the positive prompt in the ComfyUI workflow canvas. Some ideas:

   > A futuristic AMD data center, rows of glowing GPUs, cinematic lighting, 4K ultra-detail

   > High-speed particles flowing through a silicon chip, abstract tech visualization, blue and red color palette

   > A gamer at a keyboard, Radeon GPU glow, motion blur, slow motion

2. **Adjust Parameters**

   - **Resolution**: Try `720x720` for faster iteration, or `1280x704` for maximum quality.
   - **Frames**: Reduce to `49` (~2 sec) for quick tests, or keep `121` for a full 5-second video.
   - **Steps**: Try `20` for speed vs. `40` for higher quality.

3. **Save Your Workflow**

   After customizing, click **Workflows -> Save As** in the ComfyUI sidebar.

---

### Creative Themes: The AMD Vision

Create a **5-10 second video** with one of the following themes:

- **AI on AMD Instinct** — visualize powerful AI inference or training on MI300X
- **Next-Gen Compute** — represent raw compute performance, silicon, or futuristic chips
- **Gaming with Radeon** — imagine energy, motion, or light inspired by GPU gaming

---

### Submission

Please submit the following:
- Your generated video (`.mp4`)
- Your ComfyUI workflow (`.json`)
- A short write-up on your approach and prompt strategy

---

### Additional Resources

- [LTX-Video Official Docs](https://docs.comfy.org/tutorials/video/ltx/ltx-2-3)
- [LTX-Video GitHub](https://github.com/Lightricks/LTX-Video)
- [ComfyUI Documentation](https://docs.comfy.org)
- [AMD ROCm Documentation](https://rocm.docs.amd.com)
